# 05 — Ingestão dos manuais Chevrolet no ChromaDB

**Projeto:** Mão na Roda — Diagnóstico Automotivo via PLN e ML
**Equipe:** Diego Spagnuolo Sugai, Kauê Henrique Matias Alves, Leonardo Moreira dos Santos, Victor Maki Tarcha
**Orientador:** Prof. Dr. Ivan Carlos Alcântara de Oliveira

Este notebook executa a etapa de **construção da base vetorial do RAG** sobre 9 manuais Chevrolet (2011-2016). Roda uma vez só — depois disso, o pipeline integrado (`03_pipeline_integrado.ipynb`) abre a base persistida no Drive em segundos.

**Plano de execução:**
1. Setup do ambiente e montagem do Drive
2. Carregamento e extração de texto dos 9 PDFs
3. Chunking semântico com overlap
4. Enriquecimento de metadata (modelo, ano, tipo de conteúdo, sistemas mapeados)
5. Geração de embeddings com BAAI/bge-m3
6. Persistência no ChromaDB (no Drive)
7. Validação com queries de teste
8. Resumo final

**Tempo estimado:** 30-50 minutos na primeira execução (download do bge-m3 + processamento dos 9 PDFs).

---
## Seção 0 — Setup do ambiente

In [ ]:
# Dependências com versões compatíveis entre si.
# numpy<2 é necessário porque várias libs ainda não suportam numpy 2.x.
# O Colab vem com numpy 2.x — precisamos forçar downgrade.
!pip install -q "numpy<2" "chromadb>=0.5,<0.6" "sentence-transformers>=2.7,<4.0" "pypdf>=4.0,<5.0" 2>&1 | tail -3
print('Dependências instaladas.')
print()
print('IMPORTANTE: Reinicie a sessão antes de continuar.')
print('  Menu → Ambiente de execução → Reiniciar sessão (NÃO clique em "Executar tudo" depois)')
print('  Depois execute as células manualmente a partir da Seção 0 célula 2 (drive.mount)')


tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.41.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.41.1 which is incompatible.
Dependências instaladas.

IMPORTANTE: Reinicie a sessão antes de continuar.
  Menu → Ambiente de execução → Reiniciar sessão (NÃO clique em "Executar tudo" depois)
  Depois execute as células manualmente a partir da Seção 0 célula 2 (drive.mount)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path
import os

BASE = Path('/content/drive/MyDrive/Agente Mecânico')
assert BASE.exists(), f'Pasta não encontrada: {BASE}'

PDFS_DIR       = BASE / 'Dataset_Nao_Estruturado'
CHROMA_DIR     = BASE / 'chromadb'
RESULTADOS_DIR = BASE / 'resultados'

CHROMA_DIR.mkdir(parents=True, exist_ok=True)
RESULTADOS_DIR.mkdir(parents=True, exist_ok=True)

assert PDFS_DIR.exists(), f'Pasta de PDFs não encontrada: {PDFS_DIR}'

# Listar PDFs disponíveis
pdfs_disponiveis = sorted([p for p in PDFS_DIR.glob('*.pdf')])
print(f'PDFs encontrados: {len(pdfs_disponiveis)}')
for p in pdfs_disponiveis:
    tamanho_mb = p.stat().st_size / 1e6
    print(f'  {p.name}  ({tamanho_mb:.1f} MB)')

PDFs encontrados: 10
  chevrolet-astra_2011_626ca7f5d5c9abfa7faf.pdf  (6.6 MB)
  chevrolet-captiva_2013_9e8f3f3bce1ac5212f22.pdf  (9.8 MB)
  chevrolet-classic_2013_fc38e3849223329d3c67.pdf  (6.1 MB)
  chevrolet-cobalt_2013_a4d4b12277fee5cad088.pdf  (9.0 MB)
  chevrolet-corsa_2011_47cf146054a5a720606a.pdf  (3.3 MB)
  chevrolet-cruze_2013_bfb363306265299ac4ec.pdf  (1.3 MB)
  chevrolet-meriva_2011_e04a259d09af85c108fe.pdf  (8.1 MB)
  chevrolet-montana_2013_36008b04a6167761fa19.pdf  (17.1 MB)
  chevrolet-s10_2013_de6de888ddf9cde6748a.pdf  (4.8 MB)
  chevrolet-tracker_2016_ab1ddc1aecd658a6cd39.pdf  (4.2 MB)


In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponível: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')

PyTorch: 2.10.0+cu128
CUDA disponível: True
Device: Tesla T4


---
## Seção 1 — Seleção dos PDFs para ingestão

O manual do Cruze 2013 é descartado porque é manual de infotainment (sistema de navegação), não de diagnóstico mecânico. Os outros 9 manuais cobrem motor, freios, suspensão, elétrica, etc.

In [ ]:
# Mapeamento: nome do arquivo → metadata estruturada
# Formato dos nomes: chevrolet-MODELO_ANO_HASH.pdf
MANUAIS_INGESTAO = {
    'chevrolet-astra_2011':    {'modelo': 'astra',    'ano': 2011, 'idioma': 'pt-BR'},
    'chevrolet-captiva_2013':  {'modelo': 'captiva',  'ano': 2013, 'idioma': 'pt-BR'},
    'chevrolet-classic_2013':  {'modelo': 'classic',  'ano': 2013, 'idioma': 'es-AR'},
    'chevrolet-cobalt_2013':   {'modelo': 'cobalt',   'ano': 2013, 'idioma': 'pt-BR'},
    'chevrolet-corsa_2011':    {'modelo': 'corsa',    'ano': 2011, 'idioma': 'pt-BR'},
    'chevrolet-meriva_2011':   {'modelo': 'meriva',   'ano': 2011, 'idioma': 'pt-BR'},
    'chevrolet-montana_2013':  {'modelo': 'montana',  'ano': 2013, 'idioma': 'pt-BR'},
    'chevrolet-s10_2013':      {'modelo': 's10',      'ano': 2013, 'idioma': 'pt-BR'},
    'chevrolet-tracker_2016':  {'modelo': 'tracker',  'ano': 2016, 'idioma': 'pt-BR'},
}

# Resolver caminho completo de cada PDF
pdfs_para_ingerir = []
for prefixo, meta in MANUAIS_INGESTAO.items():
    encontrados = list(PDFS_DIR.glob(f'{prefixo}*.pdf'))
    if not encontrados:
        print(f' Não encontrado: {prefixo}')
        continue
    pdf_path = encontrados[0]
    pdfs_para_ingerir.append((pdf_path, meta))
    print(f' {pdf_path.name} → modelo={meta["modelo"]}, ano={meta["ano"]}')

print(f'\nTotal a ingerir: {len(pdfs_para_ingerir)} PDFs')

 chevrolet-astra_2011_626ca7f5d5c9abfa7faf.pdf → modelo=astra, ano=2011
 chevrolet-captiva_2013_9e8f3f3bce1ac5212f22.pdf → modelo=captiva, ano=2013
 chevrolet-classic_2013_fc38e3849223329d3c67.pdf → modelo=classic, ano=2013
 chevrolet-cobalt_2013_a4d4b12277fee5cad088.pdf → modelo=cobalt, ano=2013
 chevrolet-corsa_2011_47cf146054a5a720606a.pdf → modelo=corsa, ano=2011
 chevrolet-meriva_2011_e04a259d09af85c108fe.pdf → modelo=meriva, ano=2011
 chevrolet-montana_2013_36008b04a6167761fa19.pdf → modelo=montana, ano=2013
 chevrolet-s10_2013_de6de888ddf9cde6748a.pdf → modelo=s10, ano=2013
 chevrolet-tracker_2016_ab1ddc1aecd658a6cd39.pdf → modelo=tracker, ano=2016

Total a ingerir: 9 PDFs


---
## Seção 2 — Extração de texto dos PDFs

Usa `pypdf` por simplicidade e robustez (lib leve, sem dependências de sistema). Cada página vira um bloco com referência de número de página, que vai para metadata.

`unstructured.io` daria estrutura semântica melhor, mas exige `poppler` instalado e demora muito mais. Para o escopo do TCC, `pypdf` + chunking semântico por janela é suficiente.

In [ ]:
from pypdf import PdfReader
import re
import time

def extrair_paginas_pdf(pdf_path: Path) -> list:
    """Extrai texto página a página. Retorna [(num_pag, texto), ...]."""
    leitor = PdfReader(str(pdf_path))
    paginas = []
    for i, page in enumerate(leitor.pages, start=1):
        try:
            texto = page.extract_text() or ''
        except Exception as e:
            print(f'  Erro pág {i}: {e}')
            texto = ''
        # Limpa espaços excessivos
        texto = re.sub(r'\s+', ' ', texto).strip()
        if len(texto) > 30:  # ignora páginas quase vazias (capa, índices muito esparsos)
            paginas.append((i, texto))
    return paginas

# Extrai todos os PDFs
todos_documentos = []  # lista de dicts: {modelo, ano, pagina, texto}
t0 = time.time()
for pdf_path, meta in pdfs_para_ingerir:
    print(f'Extraindo: {pdf_path.name}...', end=' ')
    paginas = extrair_paginas_pdf(pdf_path)
    print(f'{len(paginas)} páginas com texto.')
    for num_pag, texto in paginas:
        todos_documentos.append({
            'modelo':  meta['modelo'],
            'ano':     meta['ano'],
            'idioma':  meta['idioma'],
            'pagina':  num_pag,
            'texto':   texto,
            'arquivo': pdf_path.name,
        })

print(f'\nTotal de páginas extraídas: {len(todos_documentos)}')
print(f'Tempo: {time.time()-t0:.1f}s')

Extraindo: chevrolet-astra_2011_626ca7f5d5c9abfa7faf.pdf... 157 páginas com texto.
Extraindo: chevrolet-captiva_2013_9e8f3f3bce1ac5212f22.pdf... 243 páginas com texto.
Extraindo: chevrolet-classic_2013_fc38e3849223329d3c67.pdf... 0 páginas com texto.
Extraindo: chevrolet-cobalt_2013_a4d4b12277fee5cad088.pdf... 273 páginas com texto.
Extraindo: chevrolet-corsa_2011_47cf146054a5a720606a.pdf... 135 páginas com texto.
Extraindo: chevrolet-meriva_2011_e04a259d09af85c108fe.pdf... 143 páginas com texto.
Extraindo: chevrolet-montana_2013_36008b04a6167761fa19.pdf... 224 páginas com texto.
Extraindo: chevrolet-s10_2013_de6de888ddf9cde6748a.pdf... 279 páginas com texto.
Extraindo: chevrolet-tracker_2016_ab1ddc1aecd658a6cd39.pdf... 258 páginas com texto.

Total de páginas extraídas: 1712
Tempo: 88.8s


---
## Seção 3 — Chunking semântico com janela deslizante

Cada página é fatiada em chunks de ~500 tokens com overlap de 50 tokens. Isso garante que conceitos que cruzam o final de um chunk apareçam no começo do próximo.

Usamos contagem por palavras (aproximação razoável de tokens em PT-BR — em média 1 palavra ≈ 1.3 tokens) por simplicidade. O limite real do bge-m3 é 8192 tokens, então estamos bem abaixo.

In [ ]:
TAMANHO_CHUNK_PALAVRAS = 350   # ~500 tokens
OVERLAP_PALAVRAS        = 35    # ~50 tokens

def chunkar_texto(texto: str, tam: int = TAMANHO_CHUNK_PALAVRAS,
                  overlap: int = OVERLAP_PALAVRAS) -> list:
    """Janela deslizante por palavras. Retorna lista de strings."""
    palavras = texto.split()
    if len(palavras) <= tam:
        return [texto]

    chunks = []
    inicio = 0
    while inicio < len(palavras):
        fim = inicio + tam
        chunk = ' '.join(palavras[inicio:fim])
        chunks.append(chunk)
        if fim >= len(palavras):
            break
        inicio += (tam - overlap)
    return chunks

# Aplica chunking em todos os documentos
chunks_todos = []  # lista de dicts: {chunk_id, texto, modelo, ano, pagina, arquivo}
for doc in todos_documentos:
    pedacos = chunkar_texto(doc['texto'])
    for j, pedaco in enumerate(pedacos):
        chunks_todos.append({
            'modelo':  doc['modelo'],
            'ano':     doc['ano'],
            'idioma':  doc['idioma'],
            'pagina':  doc['pagina'],
            'arquivo': doc['arquivo'],
            'sub':     j,
            'texto':   pedaco,
        })

print(f'Total de chunks gerados: {len(chunks_todos)}')

# Distribuição por modelo
from collections import Counter
contagem = Counter(c['modelo'] for c in chunks_todos)
print('\nChunks por modelo:')
for modelo, n in sorted(contagem.items()):
    print(f'  {modelo:12s} {n:5d} chunks')

Total de chunks gerados: 1946

Chunks por modelo:
  astra          194 chunks
  captiva        315 chunks
  cobalt         281 chunks
  corsa          160 chunks
  meriva         178 chunks
  montana        227 chunks
  s10            288 chunks
  tracker        303 chunks


---
## Seção 4 — Enriquecimento de metadata

Para cada chunk, deduzimos:

- **`tipo_conteudo`** — categoriza o tipo de informação (luz_painel, procedimento_emergencia, descricao_componente, advertencia_seguranca, manutencao_periodica, especificacao_tecnica, outro). Heurística baseada em palavras-chave.
- **`sistemas_mapeados`** — quais das 10 classes do dataset aquele chunk cobre. Permite filtrar no retrieval pela classe predita pelo BERTimbau.

Estas metadatas são essenciais para o filtro de busca da Etapa 3b do pipeline.

In [ ]:
# Heurísticas para categorizar conteúdo (palavras em PT-BR)
PALAVRAS_TIPO = {
    'luz_painel':              ['luz indicadora', 'luz acende', 'painel de instrumentos',
                                'luz de aviso', 'lampada acende', 'sinaliza', 'aviso luminoso'],
    'procedimento_emergencia': ['em caso de emergência', 'pane', 'parar imediatamente',
                                'reboque', 'guincho', 'estacionar com segurança',
                                'desligue o motor', 'não dirija'],
    'advertencia_seguranca':   ['atenção', 'perigo', 'cuidado', 'risco de',
                                'pode causar', 'ferimentos', 'morte'],
    'manutencao_periodica':    ['revisão', 'manutenção preventiva', 'cada 10.000', 'a cada 12 meses',
                                'plano de manutenção', 'troca periódica', 'verificação periódica'],
    'descricao_componente':    ['localizado', 'composto por', 'sistema de', 'consiste em',
                                'funciona', 'tem a função', 'projetado para'],
    'especificacao_tecnica':   ['litros', 'volts', 'kg', 'pressão', 'capacidade',
                                'especificação', 'ficha técnica', 'consumo'],
}

# Mapeamento sistema → palavras-chave (espelha as 10 classes do dataset)
PALAVRAS_SISTEMA = {
    0: ['bateria', 'alternador', 'partida', 'elétrica', 'fusível', 'chicote',
        'farol', 'lâmpada', 'corrente', 'motor de arranque'],
    1: ['freio', 'pastilha', 'disco', 'pinça', 'fluido de freio', 'pedal',
        'frenagem', 'abs', 'tambor'],
    2: ['superaquecimento', 'temperatura', 'ferver', 'radiador',
        'líquido de arrefecimento', 'termostato', 'ventoinha'],
    3: ['suspensão', 'amortecedor', 'mola', 'coxim', 'bandeja',
        'pivô', 'rolamento de roda', 'batente'],
    4: ['embreagem', 'transmissão', 'câmbio', 'marcha', 'engatar',
        'desengatar', 'patinar', 'volante do motor'],
    5: ['vazamento', 'óleo', 'fluido', 'gotejando', 'mancha',
        'retentor', 'junta', 'reservatório'],
    6: ['arrefecimento', 'água', 'radiador', 'mangueira', 'aditivo',
        'reservatório de expansão', 'bomba d\'água'],
    7: ['pneu', 'roda', 'calibrar', 'pressão dos pneus', 'estepe',
        'twi', 'desgaste do pneu', 'aquaplanagem'],
    8: ['injeção', 'combustível', 'gasolina', 'álcool', 'flex',
        'bico injetor', 'bomba de combustível', 'borboleta', 'sonda lambda'],
    9: ['escapamento', 'catalisador', 'silencioso', 'descarga',
        'fumaça', 'cano de escape'],
}

def classificar_tipo(texto: str) -> str:
    t = texto.lower()
    contagens = {tipo: sum(1 for kw in kws if kw in t) for tipo, kws in PALAVRAS_TIPO.items()}
    melhor = max(contagens.items(), key=lambda x: x[1])
    return melhor[0] if melhor[1] > 0 else 'outro'

def mapear_sistemas(texto: str) -> list:
    t = texto.lower()
    sistemas = []
    for sis_id, kws in PALAVRAS_SISTEMA.items():
        if any(kw in t for kw in kws):
            sistemas.append(sis_id)
    return sistemas

# Aplica em todos os chunks
for c in chunks_todos:
    c['tipo_conteudo']     = classificar_tipo(c['texto'])
    sistemas               = mapear_sistemas(c['texto'])
    # ChromaDB metadata aceita só tipos simples — guarda como string CSV
    c['sistemas_mapeados'] = ','.join(map(str, sistemas)) if sistemas else ''
    c['tem_sistema']       = bool(sistemas)

# Estatísticas
from collections import Counter
print('Distribuição de tipo_conteudo:')
for tipo, n in Counter(c['tipo_conteudo'] for c in chunks_todos).most_common():
    print(f'  {tipo:30s} {n:5d}')

print(f'\nChunks com pelo menos um sistema mapeado: '
      f'{sum(1 for c in chunks_todos if c["tem_sistema"])} '
      f'de {len(chunks_todos)} '
      f'({100*sum(1 for c in chunks_todos if c["tem_sistema"])/len(chunks_todos):.1f}%)')

Distribuição de tipo_conteudo:
  advertencia_seguranca            509
  descricao_componente             404
  outro                            309
  luz_painel                       303
  procedimento_emergencia          161
  manutencao_periodica             152
  especificacao_tecnica            108

Chunks com pelo menos um sistema mapeado: 1463 de 1946 (75.2%)


---
## Seção 5 — Geração de embeddings com BAAI/bge-m3

Modelo multilíngue, 1024 dimensões, treinado especificamente para retrieval. Primeira execução baixa ~2.2GB.

**Importante:** o modelo deve ser **o mesmo** usado depois na consulta (no `03_pipeline_integrado.ipynb`). Se trocar de modelo, precisa reingerir tudo.

In [ ]:
from sentence_transformers import SentenceTransformer

NOME_EMBED = 'BAAI/bge-m3'
print(f'Carregando {NOME_EMBED}... (primeira vez baixa ~2.2GB)')
modelo_emb = SentenceTransformer(NOME_EMBED)
print(f'Dimensão dos embeddings: {modelo_emb.get_sentence_embedding_dimension()}')
print(f'Device: {modelo_emb.device}')

Carregando BAAI/bge-m3... (primeira vez baixa ~2.2GB)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Dimensão dos embeddings: 1024
Device: cuda:0


In [ ]:
import time

textos = [c['texto'] for c in chunks_todos]
print(f'Gerando embeddings de {len(textos)} chunks...')

t0 = time.time()
embeddings = modelo_emb.encode(
    textos,
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True,    # cosseno via dot product depois
    convert_to_numpy=True,
)
print(f'\nTempo total: {time.time()-t0:.1f}s')
print(f'Shape dos embeddings: {embeddings.shape}')

Gerando embeddings de 1946 chunks...


Batches:   0%|          | 0/122 [00:00<?, ?it/s]


Tempo total: 172.4s
Shape dos embeddings: (1946, 1024)


---
## Seção 6 — Persistência no ChromaDB

ChromaDB com cliente persistente — salva direto no Drive. Coleção é dropada e recriada a cada execução do notebook (ingestão é idempotente).

In [ ]:
import chromadb

# Cliente persistente apontando para o Drive
client = chromadb.PersistentClient(path=str(CHROMA_DIR))

NOME_COLECAO = 'mao_na_roda_manuais'

# Drop coleção existente (re-ingestão limpa)
try:
    client.delete_collection(NOME_COLECAO)
    print(f'Coleção anterior removida.')
except Exception:
    pass

colecao = client.create_collection(
    name=NOME_COLECAO,
    metadata={'descricao': 'Manuais Chevrolet 2011-2016 para RAG do Mão na Roda',
              'modelo_embedding': NOME_EMBED,
              'dim_embedding': 1024},
)
print(f'Coleção criada: {NOME_COLECAO}')

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Coleção criada: mao_na_roda_manuais


In [ ]:
# Insere em lotes (ChromaDB recomenda batches de até ~5000)
TAMANHO_BATCH = 500

ids_chunks = [
    f'{c["modelo"]}_{c["ano"]}_p{c["pagina"]}_s{c["sub"]}'
    for c in chunks_todos
]

# Garantir IDs únicos (evitar duplicatas)
assert len(set(ids_chunks)) == len(ids_chunks), 'IDs duplicados detectados'

metadatas = [{
    'modelo':            c['modelo'],
    'ano':               int(c['ano']),
    'idioma':            c['idioma'],
    'pagina':            int(c['pagina']),
    'arquivo':           c['arquivo'],
    'tipo_conteudo':     c['tipo_conteudo'],
    'sistemas_mapeados': c['sistemas_mapeados'],
    'tem_sistema':       bool(c['tem_sistema']),
} for c in chunks_todos]

# Inserir em batches
import time
t0 = time.time()
n_total = len(chunks_todos)
for i in range(0, n_total, TAMANHO_BATCH):
    fim = min(i + TAMANHO_BATCH, n_total)
    colecao.add(
        ids=ids_chunks[i:fim],
        embeddings=embeddings[i:fim].tolist(),
        documents=textos[i:fim],
        metadatas=metadatas[i:fim],
    )
    print(f'  Inseridos {fim}/{n_total}')

print(f'\nIngestão concluída em {time.time()-t0:.1f}s')
print(f'Coleção tem {colecao.count()} chunks indexados.')

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


  Inseridos 500/1946
  Inseridos 1000/1946
  Inseridos 1500/1946
  Inseridos 1946/1946

Ingestão concluída em 8.6s
Coleção tem 1946 chunks indexados.


---
## Seção 7 — Validação com queries de teste

Roda 5 queries representativas das 10 classes do dataset. Para cada uma, mostra os top-3 chunks recuperados — relevância deve ser inspecionada manualmente.

In [ ]:
QUERIES_TESTE = [
    ('carro não pega de manhã, só dá um clique',           [0]),
    ('pedal do freio afundou totalmente',                  [1]),
    ('temperatura subiu até o vermelho, motor fervendo',   [2, 6]),
    ('carro fazendo barulho ao passar em buraco',          [3]),
    ('pneu murcho depois de pegar prego',                  [7]),
]

def buscar_chunks(query: str, top_k: int = 3, sistemas_alvo: list = None):
    """Busca chunks relevantes, opcionalmente filtrando por sistemas."""
    q_emb = modelo_emb.encode(query, normalize_embeddings=True).tolist()

    # Sem filtro estrito — pegamos top_k livremente e mostramos sistemas
    res = colecao.query(
        query_embeddings=[q_emb],
        n_results=top_k,
    )
    return res

print('VALIDAÇÃO DO RETRIEVAL')
print('='*70)
for query, sistemas_esperados in QUERIES_TESTE:
    print(f'\nQuery: "{query}"')
    print(f'  Sistemas esperados: {sistemas_esperados}')
    res = buscar_chunks(query, top_k=3)

    for i in range(len(res['documents'][0])):
        meta = res['metadatas'][0][i]
        dist = res['distances'][0][i]
        doc  = res['documents'][0][i]
        sis  = meta['sistemas_mapeados'] or '(nenhum)'
        print(f'  [#{i+1}] dist={dist:.3f} | {meta["modelo"]} {meta["ano"]} '
              f'pág.{meta["pagina"]} | tipo={meta["tipo_conteudo"]} | sistemas={sis}')
        # Mostra os primeiros 150 caracteres do chunk
        print(f'        "{doc[:150]}..."')

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


VALIDAÇÃO DO RETRIEVAL

Query: "carro não pega de manhã, só dá um clique"
  Sistemas esperados: [0]
  [#1] dist=0.942 | captiva 2013 pág.188 | tipo=descricao_componente | sistemas=0,2,4,7
        "CONFORTO E CONVENIÊNCIA 10-24 Captiva Sport, 03/13 SEÇÃO 10 Como o sistema funciona O sistema de detecção de objetos irá funci- onar automaticamente q..."
  [#2] dist=0.968 | meriva 2011 pág.55 | tipo=descricao_componente | sistemas=(nenhum)
        "funciona com o motor em funcionamento. Para maioreficiência do sistema, os vidros devemestar fechados. Caso o interior do veículo tenha se aquecido de..."
  [#3] dist=0.971 | tracker 2016 pág.147 | tipo=advertencia_seguranca | sistemas=0,1,4
        "&KHYUROHW7UDFNHU %UD]LO 2ZQHU0DQXDO *06$/RFDOL]LQJ%UD]LO    FUF 144 Condução e operação mais automática nem manual..."

Query: "pedal do freio afundou totalmente"
  Sistemas esperados: [1]
  [#1] dist=0.872 | meriva 2011 pág.60 | tipo=descricao_componente | sistema

---
## Seção 8 — Teste de filtros de metadata

Demonstra como o pipeline integrado vai usar filtros para restringir a busca por modelo/ano/sistema. Esse é o padrão que o `03_pipeline_integrado.ipynb` segue na Etapa 3b.

In [ ]:
# Exemplo: usuário tem Corsa 2011, BERTimbau classificou como classe 1 (freios)
print('Exemplo: relato de freio em Corsa 2011')
print('='*70)

q = 'pedal do freio está duro e o carro não para'
q_emb = modelo_emb.encode(q, normalize_embeddings=True).tolist()

# Filtro: só chunks do Corsa, ano próximo a 2011, com sistema 1 mapeado
res = colecao.query(
    query_embeddings=[q_emb],
    n_results=5,
    where={
        '$and': [
            {'modelo':       {'$eq': 'corsa'}},
            {'tem_sistema':  {'$eq': True}},
        ]
    },
)

print(f'\nQuery: "{q}"')
print(f'Filtros: modelo=corsa AND tem_sistema=True\n')

if not res['documents'][0]:
    print('  Nenhum chunk encontrado com esse filtro.')
else:
    for i in range(len(res['documents'][0])):
        meta = res['metadatas'][0][i]
        dist = res['distances'][0][i]
        sis  = meta['sistemas_mapeados']
        # Filtra os que têm classe 1 (freios) na lista de sistemas
        sistemas_int = [int(x) for x in sis.split(',') if x]
        marca_freio  = ' (FREIO!)' if 1 in sistemas_int else ''
        print(f'  [#{i+1}] dist={dist:.3f} | pág.{meta["pagina"]} | '
              f'sistemas=[{sis}]{marca_freio}')
        print(f'        "{res["documents"][0][i][:140]}..."')

Exemplo: relato de freio em Corsa 2011

Query: "pedal do freio está duro e o carro não para"
Filtros: modelo=corsa AND tem_sistema=True

  [#1] dist=0.704 | pág.51 | sistemas=[1,4,7] (FREIO!)
        "COMANDOS E CONTROLES Corsa, 03/10 6-31 SEÇÃO 6 Frenagem de emergência Quase todo motorista já enfrentou alguma situação em que precisou de f..."
  [#2] dist=0.832 | pág.50 | sistemas=[1,5,7] (FREIO!)
        "COMANDOS E CONTROLES 6-30 Corsa, 03/10 SEÇÃO 6 Freio de estacionamento O freio de estacionamento atua mecanica- mente nas rodas traseiras e ..."
  [#3] dist=0.931 | pág.115 | sistemas=[1,4,5,7,8] (FREIO!)
        "SERVIÇOS E MANUTENÇÃO 13-8 Corsa, 03/10 SEÇÃO 13 Freios Fluido de freio Verifique o nível do fluido mensalmente ou quando acender a luz indi..."
  [#4] dist=0.942 | pág.32 | sistemas=[0]
        "esta-ções do mesmo tipo, e não pode causar interferência a sistemas operando em caráter primário. Nunca dê partidas contí-nuas no motor por ..."
  [#5] dist=0.952 | pág.67 | sistem

---
## Seção 9 — Sumário e próximos passos

In [ ]:
print('INGESTÃO CONCLUÍDA')
print('='*70)
print(f'\nColeção:       {NOME_COLECAO}')
print(f'Chunks:        {colecao.count()}')
print(f'Modelo embed:  {NOME_EMBED} ({modelo_emb.get_sentence_embedding_dimension()}d)')
print(f'Persistência:  {CHROMA_DIR}')

# Tamanho em disco
tamanho_mb = sum(p.stat().st_size for p in CHROMA_DIR.rglob('*') if p.is_file()) / 1e6
print(f'Tamanho disco: {tamanho_mb:.0f} MB')

print('\nPróximo passo:')
print('  → 03_pipeline_integrado.ipynb detecta a coleção automaticamente')
print('  → Etapa 3b (RAG) usa os chunks indexados aqui')
print('  → Re-ingestão só é necessária se a planilha de manuais mudar')

INGESTÃO CONCLUÍDA

Coleção:       mao_na_roda_manuais
Chunks:        1946
Modelo embed:  BAAI/bge-m3 (1024d)
Persistência:  /content/drive/MyDrive/Agente Mecânico/chromadb
Tamanho disco: 33 MB

Próximo passo:
  → 03_pipeline_integrado.ipynb detecta a coleção automaticamente
  → Etapa 3b (RAG) usa os chunks indexados aqui
  → Re-ingestão só é necessária se a planilha de manuais mudar
